## ⚙️ Environment setup (one-time step)

If you are running this notebook for the first time, execute the next cell to install:
- **PuLP** — the optimization library we will use
- **Matplotlib** — for simple visualizations later

In [ ]:
# install pulp if needed
%pip install pulp
%pip install matplotlib

# 🍕🍔 Feeding 100 People — A Gentle Introduction to Optimization

## What is this notebook about?
In this notebook, we will learn how **optimization** helps make better decisions.

We will try to answer a very concrete question:

> **How many meals of each type should we order to feed 100 people, respect a limited budget, and keep people as happy as possible?**

No background in optimization or mathematics is required.

---

## Step 0 — The story (no math, no code)

You are organizing dinner for **100 people**.

You can order the following meals:
- Pizza (Pepperoni or Vegetarian)
- Burger
- Fish & dips
- Greek Souvlaki
- Moussaka

Each meal:
- Has a **cost**
- Brings a certain level of **happiness**

Your goal is simple:

> **Choose the quantities of each meal to maximize happiness while staying within the budget.**

---

### 🧠 No code in this step
This step is only about understanding the problem.

Take a moment to think:
- What decisions do we need to make?
- What limits our choices?

---

---
## Step 1 — Basic data (costs, happiness, budget)

We start with a **very simple assumption**:
each meal has an average happiness score.

You can think of this as the result of a quick internal survey.

At this stage, we do not worry about individual preferences — we just want to understand the logic of the model.

---

In [ ]:
### 💻 Code — Define the data
# Define meal types, costs, happiness scores and global constraints

# Cost per meal (in euros)
cost = {
    "pizza": 8,
    "burger": 10,
    "fish": 12,
    "souvlaki": 11,
    "moussaka": 9
}

# Happiness score per meal (totally subjective 😄)
base_happiness = {
    "pizza": 7,
    "burger": 8,
    "fish": 9,
    "souvlaki": 8,
    "moussaka": 7
}

# Total budget
base_budget = 900

# Number of people to feed
base_nbr_people = 100

## 🧰 Helper class: `DinnerPlanner` (no need to modify)

The next code cell defines a **helper class** called `DinnerPlanner`.

This class:
- Initialize the optimization model
- Solves it using a linear programming solver
- Displays the results in a readable way
- Plots a simple visualization of the meal quantities

👉 **You do not need to understand or modify this code** to follow the rest of the notebook.

You can use this class by calling it with the problem parameters: **DinnerPlanner(budget, nb_people, cost, happiness)**

✅ Just run the next cell  
✅ We’ll interact with it through simple, high-level calls later

Feel free to come back to this cell at the end if you’re curious 🙂


In [ ]:
from pulp import LpProblem, LpMaximize, LpVariable
import matplotlib.pyplot as plt

class DinnerPlanner:

  def __init__(self, budget: int = base_budget, nb_people: int = base_nb_people, cost: dict[str, int] = cost, happiness: dict[str, int] = base_happiness):
    self.budget: int = budget
    self.nb_people: int = nb_people
    self.cost: dict[str, int] = cost
    self.happiness: dict[str, int] = base_happiness
    self.model: LpProblem =  LpProblem("Dinner_Planning", LpMaximize)
    self.variables: dict[str, LpVariable] | None = None

  def solve(self, show = True):
    solved = self.model.solve()
    if show :
      if solved != 1 :
        print("No feasible solution")
      print("Optimal meal plan:\n")

      for m in self.variables:
        print(f"{m}: {self.variables[m].value():.1f}")

      print("\nTotal cost:",
      sum(self.cost[m] * self.variables[m].value() for m in self.variables))

      print("Total happiness:",
      sum(self.happiness[m] * self.variables[m].value() for m in self.variables))
    if solved == 1 :
      self.mean_happiness = sum(self.happiness[m] * self.variables[m].value() for m in self.variables) / self.nb_people
    else :
      self.mean_happiness = 0

  def plot_histogram(self):
    values = list(self.variables.values())
    names = list(self.variables.keys())
    values = [v.value() for v in values]

    # Use a modern and visually appealing color palette
    colors = plt.cm.viridis([i / len(names) for i in range(len(names))])

    plt.figure(figsize=(12, 7))
    bars = plt.bar(names, values, color=colors, edgecolor="#333333", linewidth=1.2)

    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2., height,
                 f'{height}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Customize the plot
    plt.title("Quantity per Option", fontsize=16, fontweight='bold', pad=20)
    plt.ylabel("Number of Meals", fontsize=14, labelpad=10)
    plt.xlabel("Options", fontsize=14, labelpad=10)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(axis="y", alpha=0.4, linestyle='--', color='#cccccc')

    # Remove top and right spines for a cleaner look
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()



---

## Step 2 — The optimization model

Now we translate the story into decisions.

### What do we decide?
We decide **how many meals of each type to order**.

### What do we want?
We want to **maximize total happiness**.

### What are the constraints?
- We cannot exceed the available budget
- We must order enough meals for every one

This is the core of optimization:
**make the best possible decision under constraints**.

---

In [ ]:
def build_variables(dinner_planner: DinnerPlanner) -> None:
    """
    Create the decision variables for the optimization model and store them
    in the DinnerPlanner instance.

    Each variable represents the number of meals to order for a given option
    (e.g., pizza, burger, fish, etc.).

    Parameters
    ----------
    dinner_planner : DinnerPlanner
        Instance containing the problem data (budget, number of people, costs).

    Notes
    -----
    The variables are stored in the `dinner_planner.variables` attribute
    and do not need to be returned.
    """
    # Variable 'x' € [0, 10]: LpVariable('x', lowBound=0, upBound=10, cat='Continuous')
    meals = {
        name: LpVariable(name, lowBound=0, cat="Continuous")
        for name in dinner_planner.cost
    }
    dinner_planner.variables = meals

In [ ]:
def build_objective(dinner_planner: DinnerPlanner):
    """
    Build the LP objective: maximize total happiness.
    """
    # The syntax: model += 2 * x, "ObjectiveName"
    # LpMaximize is set when we created the model in DinnerPlanner
    # hint: You should use variables defined previously in dinner_planner.variables
    dinner_planner.model += sum(dinner_planner.hapiness[m] * dinner_planner.variables[m] for m in dinner_planner.cost), "Total_Happiness"

In [ ]:
def build_constraints(dinner_planner: DinnerPlanner):
    """
    Add LP constraints using the syntax:
    model += linear_expression <= or >= value, "Constraint_Name"
    """
    # assert we respect the budget
    dinner_planner.model += sum(dinner_planner.cost[m] * dinner_planner.variables[m] for m in dinner_planner.variables) <= dinner_planner.budget, "Budget_Constraint"
    # define a limit of meals that can be ordered per person
    dinner_planner.model += sum(dinner_planner.variables[m] for m in dinner_planner.variables) <= 2*dinner_planner.nbr_people, "Meal_Limit_Max"
    # make sure everyone can be fed
    dinner_planner.model += sum(dinner_planner.variables[m] for m in dinner_planner.variables) >= dinner_planner.nbr_people, "Feed_everyone"

In [ ]:
def build_model(dinner_planner: DinnerPlanner):
    """
    Build the complete optimization model.

    This function orchestrates the building process by calling:
    1. `build_variables` to define the decision variables
    2. `build_objective` to define the objective function
    3. `build_constraints` to add all necessary constraints

    Parameters
    ----------
    dinner_planner : DinnerPlanner
        Instance containing the problem data (budget, number of people, costs, happiness, etc.)

    Notes
    -----
    After calling this function, `dinner_planner.model` is fully defined
    and ready to be solved.
    """
    build_variables(dinner_planner)
    build_objective(dinner_planner)
    build_constraints(dinner_planner)

---

## Results and interpretation

Once the model is solved, we obtain:
- The number of meals of each type to order
- A solution that respects all constraints
- A plan that maximizes happiness according to our assumptions

At this point, the solution may already feel surprisingly reasonable.

---

In [ ]:
# Solve the model
dinner_planner = DinnerPlanner()

build_model(dinner_planner)

dinner_planner.solve()

dinner_planner.plot_histogram()

## Observation
- Did you expect this result ?
- Do we really need a solver ?

Let's play with the parameters

In [ ]:
dinner_planner = DinnerPlanner(budget = 1100, nb_people = 70)

build_model(dinner_planner)

dinner_planner.solve()

dinner_planner.plot_histogram()

In [ ]:

def plot_happiness_budget_based(budgets):
  nb_people = 100
  mean_happiness = []
  for budget in budgets :
    dinner_planner = DinnerPlanner(budget = budget, nb_people = nb_people)
    build_variables(dinner_planner)
    build_objective(dinner_planner)
    build_constraints(dinner_planner)
    dinner_planner.solve(show = False)
    mean_happiness.append(dinner_planner.mean_happiness)

  plt.figure(figsize=(12, 7), facecolor='#f9f9f9')
  plt.plot(budgets, mean_happiness, marker='o', markersize=10, linestyle='-', linewidth=2.5, color='#2ecc71', label='Mean Happiness')

  # Fill the area under the curve for a modern look
  plt.fill_between(budgets, mean_happiness, color='#2ecc71', alpha=0.15)

  # Customize the plot
  plt.title("Mean Happiness vs. Budget", fontsize=18, fontweight='bold', pad=20, color='#333333')
  plt.xlabel("Budget (€)", fontsize=14, labelpad=10, color='#333333')
  plt.ylabel("Mean Happiness", fontsize=14, labelpad=10, color='#333333')
  plt.xticks(fontsize=12, color='#333333')
  plt.yticks(fontsize=12, color='#333333')
  plt.grid(True, linestyle='--', alpha=0.5, color='#cccccc')

  # Add a subtle background and remove top/right spines
  plt.gca().set_facecolor('#f9f9f9')
  plt.gca().spines['top'].set_visible(False)
  plt.gca().spines['right'].set_visible(False)
  plt.gca().spines['left'].set_color('#cccccc')
  plt.gca().spines['bottom'].set_color('#cccccc')

  # Add a legend
  plt.legend(fontsize=12, facecolor='#f9f9f9', edgecolor='#f9f9f9')

  # Annotate the plot with the maximum happiness
  max_happiness = max(mean_happiness)
  max_budget = budgets[mean_happiness.index(max_happiness)]
  plt.annotate(f'Max Happiness: {max_happiness:.2f}',
             xy=(max_budget, max_happiness),
             xytext=(max_budget + 10, max_happiness - 0.05),
             arrowprops=dict(facecolor='#e74c3c', shrink=0.05, width=1, headwidth=5),
             fontsize=12, color='#e74c3c', fontweight='bold')

  plt.tight_layout()
  plt.show()


budgets = [200 * i for i in range(1, 26)]
plot_happiness_budget_based(budgets)

## What can you say of this plot ?





---

## ⭐ Bonus 1 — Individual preferences (more realism)

In reality, people do not all like the same food.

Some love pizza, others prefer fish, and some really want Greek food.

To reflect this, we can:
- Assign a preference score to each person for each meal

This keeps the model simple while making it more realistic.

The optimization logic does not change — only the data does.

---

In [ ]:
import random
import numpy as np
from typing import Dict, List

random.seed(42)
num_people = 100

def init_happiness(nb_people: int, meals_name: List[str]) -> Dict[int, Dict[str, int]]:
    """
    Initialize individual happiness preferences score (1-10) for each person.

    Parameters
    ----------
    nb_people : int
        Number of people to generate preferences for.
    meals_name : List[str]
        List of meal type names.

    Returns
    -------
    Dict[int, Dict[str, int]]
        Dictionary mapping person ID to a dictionary of meal preferences.
        Example: {0: {'pizza': 7, 'burger': 3, ...}, 1: {...}, ...}
    """
    preferences = {
        p: {
            meal: random.randint(1, 10)
            for meal in meals_name
        }
        for p in range(nb_people)
    }
    return preferences

happiness_per_person = init_happiness(num_people, list(cost.keys()))

In [ ]:

from pulp import LpProblem, LpMaximize, LpVariable
import matplotlib.pyplot as plt
from typing import Dict, Optional


class DinnerPlanner:

  def __init__(
      self,
      budget: int,
      nb_people: int,
      happiness_per_person: Dict[int, Dict[str, int]] = happiness_per_person,
      cost: Dict[str, int] = cost
  ):
    self.budget: int = budget
    self.nb_people: int = nb_people
    self.cost: Dict[str, int] = cost
    self.happiness_per_person: Dict[int, Dict[str, int]] = happiness_per_person
    self.model: LpProblem = LpProblem("Dinner_Planning", LpMaximize)
    self.variables: Optional[Dict[str, LpVariable]] = None
    self.variables_per_person: Optional[Dict[str, Dict[int, LpVariable]]] = None

  def solve(self, show: bool = True) -> None:
    solved = self.model.solve()
    if solved == 1:
      total_happiness = sum(
        sum(
          self.happiness_per_person[p][m] * self.variables_per_person[m][p].value()
          for m in self.cost
        )
        for p in range(self.nb_people)
      )
      self.mean_happiness = total_happiness / self.nb_people
    else:
      self.mean_happiness = 0
      total_happiness = 0

    if show:
      if solved != 1:
        print("No feasible solution")
        return
      print("Optimal meal plan:\n")

      for m in self.variables:
        print(f"{m}: {self.variables[m].value():.1f}")

      print("\nTotal cost:",
      sum(self.cost[m] * self.variables[m].value() for m in self.variables))

      print("Total happiness:",
      total_happiness)

  def plot_histogram(self) -> None:
    values = list(self.variables.values())
    names = list(self.variables.keys())
    values = [v.value() for v in values]

    # Use a modern and visually appealing color palette
    colors = plt.cm.viridis([i / len(names) for i in range(len(names))])

    plt.figure(figsize=(12, 7))
    bars = plt.bar(names, values, color=colors, edgecolor="#333333", linewidth=1.2)

    # Add value labels on top of each bar
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2., height,
                 f'{height}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Customize the plot
    plt.title("Quantity per Option", fontsize=16, fontweight='bold', pad=20)
    plt.ylabel("Number of Meals", fontsize=14, labelpad=10)
    plt.xlabel("Options", fontsize=14, labelpad=10)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(axis="y", alpha=0.4, linestyle='--', color='#cccccc')

    # Remove top and right spines for a cleaner look
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
# we want global variables Dict[meal_name, LpVariable] the total number of meals ordered
# we want variables per person Dict[meal_name, Dict[person_id, LpVariable]] the number of meals ordered per person
def build_variables(dinner_planner: DinnerPlanner) -> None:
    """
    Create decision variables for the optimization model with individual preferences.

    Creates two types of variables:
    1. Global variables: total number of each meal type ordered
    2. Per-person variables: number of each meal type assigned to each person

    Parameters
    ----------
    dinner_planner : DinnerPlanner
        Instance containing the problem data (budget, number of people, costs, happiness_per_person).

    Notes
    -----
    The variables are stored in:
    - `dinner_planner.variables`: Dict[meal_name, LpVariable] for global meal counts
    - `dinner_planner.variables_per_person`: Dict[meal_name, Dict[person_id, LpVariable]] for per-person assignments
    """
    # Global variables: total number of meals of each type
    meals_global = {
        name: LpVariable(name, lowBound=0, cat="Continuous")
        for name in dinner_planner.cost
    }
    dinner_planner.variables = meals_global

    # Per-person variables: number of meals of each type for each person
    meals_per_person = {
        name: LpVariable.dicts(name,indices = range(dinner_planner.nbr_people), lowBound=0, cat="Continuous")
        for name in dinner_planner.cost
    }
    dinner_planner.variables_per_person = meals_per_person



In [ ]:
#redefine the happiness based on individual preferences
def build_objective(dinner_planner: DinnerPlanner) -> None:
    """
    Build the LP objective: maximize total happiness based on individual preferences.

    The objective sums up the happiness for each person-meal combination,
    weighted by the person's preference for that meal type.

    Parameters
    ----------
    dinner_planner : DinnerPlanner
        Instance containing the problem data and variables.
    """
    # Sum happiness across all people and all meals
    # For each person p and meal m, add: happiness_per_person[p][m] * variables_per_person[m][p]
    total_happiness = sum(
        sum(
            dinner_planner.preferences[p][m] * dinner_planner.variables_per_person[m][p] 
            for m in dinner_planner.cost) for p in range(dinner_planner.nbr_people
        )
    )
    dinner_planner.model += total_happiness, "Total_Happiness"

In [ ]:
def build_constraints(dinner_planner: DinnerPlanner) -> None:
    """
    Add LP constraints for the individual preferences model.

    Constraints include:
    1. Budget constraint: total cost cannot exceed budget
    2. Linking constraints: global variables must equal sum of per-person variables
    3. Per-person constraints: each person must get at least 1 meal and at most 3 meals

    Parameters
    ----------
    dinner_planner : DinnerPlanner
        Instance containing the problem data and variables.
    """
    # The budget constraint (same as before :p)
    dinner_planner.model += sum(
        dinner_planner.cost[m] * dinner_planner.variables[m] for m in dinner_planner.variables
        ) <= dinner_planner.budget, "Budget_Constraint"

    # We need to link global and per-person variables
    # For each meal type, the global variable must equal the sum of all per-person variables
    for m in dinner_planner.variables:
        dinner_planner.model += sum(
            dinner_planner.variables_per_person[m][p] for p in range(dinner_planner.nbr_people)
            ) == dinner_planner.variables[m], f"Link_Constraint_{m}"

    # Define limits for per person to eat
    # Each person must get at least 1 meal and at most 3 meals
    for p in range(dinner_planner.nb_people):
        dinner_planner.model += sum(dinner_planner.variables_per_person[m][p] for m in dinner_planner.variables) <= 3, f"Meal_Limit_Max_{p}"
        dinner_planner.model += sum(dinner_planner.variables_per_person[m][p] for m in dinner_planner.variables) >= 1, f"Meal_Limit_Min_{p}"

In [ ]:
# Solve the model
dinner_planner = DinnerPlanner(budget = 900, nb_people = 100)

build_model(dinner_planner)

dinner_planner.solve()

dinner_planner.plot_histogram()

In [ ]:
budgets = [200 * i for i in range(1, 21)]
plot_happiness_budget_based(budgets)

---

## ⭐ Bonus 2 — Integer decisions (real-life constraint)

So far, the model may suggest ordering:
> 123.6 pizzas

In real life, meals come in whole units.

By forcing decision variables to be integers, we make the solution:
- More realistic
- Slightly harder to compute
- Much closer to real-world optimization problems

---

In [ ]:
def build_variables(dinner_planner: DinnerPlanner) -> None:
    """
    Create integer decision variables for the optimization model.

    This version uses integer variables instead of continuous ones,
    making the solution more realistic (you can't order 123.6 pizzas).

    Creates two types of variables:
    1. Global variables: total number of each meal type ordered (integer)
    2. Per-person variables: number of each meal type assigned to each person (integer)

    Parameters
    ----------
    dinner_planner : DinnerPlanner
        Instance containing the problem data (budget, number of people, costs, happiness_per_person).

    Notes
    -----
    The variables are stored in:
    - `dinner_planner.variables`: Dict[meal_name, LpVariable] for global meal counts (integer)
    - `dinner_planner.variables_per_person`: Dict[meal_name, Dict[person_id, LpVariable]] for per-person assignments (integer)
    """
    # Global variables: total number of meals of each type (integer)
    # LpVariable(name, lowBound=0, cat='Integer')
    meals_global = {
        name: LpVariable(name, lowBound=0, cat="Continuous")
        for name in dinner_planner.cost
    }
    dinner_planner.variables = meals_global

    # Per-person variables: number of meals of each type for each person (integer)
    meals_per_person = {
        name: {
            p: LpVariable.dicts(name,indices = range(dinner_planner.nbr_people), lowBound=0, cat="Integer")
            for p in range(dinner_planner.nb_people)
        }
        for name in dinner_planner.cost
    }
    dinner_planner.variables_per_person = meals_per_person

In [ ]:
# Solve the model
dinner_planner = DinnerPlanner(budget = 900, nb_people = 100)

build_model(dinner_planner)

dinner_planner.solve()

dinner_planner.plot_histogram()

In [ ]:
budgets = [200 * i for i in range(1, 21)]
plot_happiness_budget_based(budgets)

With the introduction of integer variables it is also possible to model some activation constraints. For instance the cook might accept to cook moussaka only if at least 50 are ordered.

In [ ]:
def build_constraints(dinner_planner: DinnerPlanner) -> None:
    """
    Add LP constraints including an activation constraint for moussaka.

    Constraints include:
    1. Budget constraint: total cost cannot exceed budget
    2. Linking constraints: global variables must equal sum of per-person variables
    3. Per-person constraints: each person must get at least 1 meal and at most 3 meals
    4. Activation constraint: moussaka can only be ordered if at least 50 are ordered

    Parameters
    ----------
    dinner_planner : DinnerPlanner
        Instance containing the problem data and variables.

    Notes
    -----
    The activation constraint uses a "big M" approach:
    - If moussaka is ordered, it must be at least 50
    - This is modeled using a binary variable: activate_moussaka
    - If activate_moussaka = 1, then moussaka >= 50
    - If activate_moussaka = 0, then moussaka = 0
    - We use: moussaka >= 50 * activate_moussaka and moussaka <= M * activate_moussaka
      (where M is a large number representing the maximum possible meals)
    """
    # Budget constraint
    dinner_planner.model += (
        sum(dinner_planner.cost[m] * dinner_planner.variables[m] for m in dinner_planner.variables) <= dinner_planner.budget,
        "Budget_Constraint"
    )

    # Linking constraints: global variables must equal sum of per-person variables
    for m in dinner_planner.variables:
        dinner_planner.model += (
            sum(dinner_planner.variables_per_person[m][p] for p in range(dinner_planner.nbr_people)) == dinner_planner.variables[m],
            f"Link_Constraint_{m}"
        )

    # Per-person constraints: each person must get at least 1 meal and at most 3 meals
    for p in range(dinner_planner.nb_people):
        dinner_planner.model += (
            sum(dinner_planner.variables_per_person[m][p] for m in dinner_planner.variables) <= 3,
            f"Meal_Limit_Max_{p}"
        )
        dinner_planner.model += (
            sum(dinner_planner.variables_per_person[m][p] for m in dinner_planner.variables) >= 1,
            f"Meal_Limit_Min_{p}"
        )

    # Activation constraint: moussaka can only be ordered if at least 50 are ordered
    # We use a binary variable to model this: if activate_moussaka = 1, then moussaka >= 50
    # If activate_moussaka = 0, then moussaka = 0
    activate_moussaka = LpVariable('activate_moussaka', cat='Binary')
    M = dinner_planner.nb_people * 3  # Large number: maximum possible meals

    # If activate_moussaka = 1, then moussaka >= 50
    dinner_planner.model += (
        dinner_planner.variables["moussaka"] >= 50 * activate_moussaka, "Moussaka_Min_Activation"
    )

    # If activate_moussaka = 0, then moussaka = 0 (enforced by: moussaka <= M * activate_moussaka)
    dinner_planner.model += (
        activate_moussaka*dinner_planner.nbr_people*3 >= dinner_planner.variables["moussaka"], "Moussaka_Max_Activation"
    )

In [ ]:
# Solve the model
dinner_planner = DinnerPlanner(budget = 900, nb_people = 100)


build_model(dinner_planner)

dinner_planner.solve()

dinner_planner.plot_histogram()